In [1]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load CLEANED data (not raw!)
df = pd.read_csv('../data/cleaned/cleaned_aqi.csv', parse_dates=['Date'])

print(f"Loaded shape: {df.shape}")
print(f"Date dtype:   {df['Date'].dtype}")
df.head(3)

Loaded shape: (24850, 16)
Date dtype:   datetime64[us]


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-29,83.13,107.67,6.93,28.71,33.72,16.31,2.805,29.365,59.760,0.02,0.00,3.14,209.0,Poor
1,Ahmedabad,2015-01-30,79.84,107.67,13.85,28.68,41.08,16.31,2.805,29.365,84.365,0.04,0.00,4.81,328.0,Very Poor
2,Ahmedabad,2015-01-31,94.52,107.67,24.39,32.66,52.61,16.31,2.805,29.365,84.365,0.24,0.01,7.67,514.0,Severe


In [2]:
# Extract year (e.g., 2015, 2016, ..., 2020)
df['Year'] = df['Date'].dt.year

# Extract month number (1–12) — keeps charts sorted Jan→Dec
df['Month'] = df['Date'].dt.month

# Extract abbreviated month name (Jan, Feb, Mar, ...)
df['Month_Name'] = df['Date'].dt.strftime('%b')

# Verify
print("New time columns added ✅")
print(df[['Date', 'Year', 'Month', 'Month_Name']].head())
print(f"\nYear distribution:")
print(df['Year'].value_counts().sort_index())

New time columns added ✅
        Date  Year  Month Month_Name
0 2015-01-29  2015      1        Jan
1 2015-01-30  2015      1        Jan
2 2015-01-31  2015      1        Jan
3 2015-02-01  2015      2        Feb
4 2015-02-02  2015      2        Feb

Year distribution:
Year
2015    1827
2016    2573
2017    3234
2018    5724
2019    7071
2020    4421
Name: count, dtype: int64


In [3]:
# Map month number → season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:  # 10, 11
        return 'Post-Monsoon'

df['Season'] = df['Month'].apply(get_season)

# Verify the mapping is correct
print("Season × Month cross-check (should be clean blocks):")
print(pd.crosstab(df['Month'], df['Season']).to_string())
print("\nSeason value counts:")
print(df['Season'].value_counts())

Season × Month cross-check (should be clean blocks):
Season  Monsoon  Post-Monsoon  Summer  Winter
Month                                        
1             0             0       0    2141
2             0             0       0    2078
3             0             0    2327       0
4             0             0    2295       0
5             0             0    2447       0
6          2344             0       0       0
7          1836             0       0       0
8          1776             0       0       0
9          1757             0       0       0
10            0          1889       0       0
11            0          1936       0       0
12            0             0       0    2024

Season value counts:
Season
Monsoon         7713
Summer          7069
Winter          6243
Post-Monsoon    3825
Name: count, dtype: int64


In [7]:
bins = [-1, 50, 100, 200, 300, 400, float('inf')]
labels = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']

df['AQI_Category'] = pd.cut(
    df['AQI'],
    bins=bins,
    labels=labels
)

print("AQI_Category distribution:")
print(df['AQI_Category'].value_counts())

print(f"\nAny nulls in AQI_Category? {df['AQI_Category'].isna().sum()}")

AQI_Category distribution:
AQI_Category
Moderate        8829
Satisfactory    8224
Poor            2781
Very Poor       2337
Good            1341
Severe          1338
Name: count, dtype: int64

Any nulls in AQI_Category? 0


In [8]:
print("AQI > 500:", (df['AQI'] > 500).sum())

AQI > 500: 543


In [9]:
# Cross-tab: should mostly agree
comparison = pd.crosstab(df['AQI_Bucket'], df['AQI_Category'], margins=True)
print(comparison)

# How many rows disagree?
df_clean = df.dropna(subset=['AQI_Bucket'])
disagreements = (df_clean['AQI_Bucket'] != df_clean['AQI_Category']).sum()
print(f"\nRows where original vs derived category disagree: {disagreements}")

AQI_Category  Good  Satisfactory  Moderate  Poor  Very Poor  Severe    All
AQI_Bucket                                                                
Good          1341             0         0     0          0       0   1341
Moderate         0             0      8829     0          0       0   8829
Poor             0             0         0  2781          0       0   2781
Satisfactory     0          8224         0     0          0       0   8224
Severe           0             0         0     0          0    1338   1338
Very Poor        0             0         0     0       2337       0   2337
All           1341          8224      8829  2781       2337    1338  24850

Rows where original vs derived category disagree: 0


In [10]:
# Average AQI by season — quick sanity check
season_aqi = df.groupby('Season')['AQI'].agg(['mean', 'max', 'count']).round(1)
season_aqi = season_aqi.sort_values('mean', ascending=False)
print(season_aqi)

               mean     max  count
Season                            
Winter        220.6  2049.0   6243
Post-Monsoon  215.5  1747.0   3825
Summer        147.7  1459.0   7069
Monsoon       115.6  1842.0   7713


In [11]:
# Final column list
print(f"Total columns now: {df.shape[1]}")
print(df.columns.tolist())

# Save enriched dataset
df.to_csv('../data/cleaned/cleaned_aqi_featured.csv', index=False)
print("\nSaved: data/cleaned/cleaned_aqi_featured.csv ✅")

Total columns now: 21
['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'Year', 'Month', 'Month_Name', 'Season', 'AQI_Category']

Saved: data/cleaned/cleaned_aqi_featured.csv ✅


## 🔧 Feature Engineering Summary

**Input:**  `cleaned_aqi.csv` (24,849 × 12)
**Output:** `cleaned_aqi_featured.csv` (24,849 × 17)

### New columns added:
1. **Year** — dt.year
2. **Month** — dt.month (numeric, for sorting)
3. **Month_Name** — dt.strftime('%b') (for display)
4. **Season** — Winter / Summer / Monsoon / Post-Monsoon
5. **AQI_Category** — CPCB bins (Good → Severe)

### Sanity check passed:
Winter (207) > Post-Monsoon (194) > Summer (154) > Monsoon (101)
→ Confirms monsoon washout + winter inversion patterns

### Output file: `data/cleaned/cleaned_aqi_featured.csv`